# 강판 표면 결함: ET + LightGBM 성능 개선 검증

**목적:** `steel.ipynb`에서 유망했던 ET+LightGBM 스태킹과 (1) 50:50 확률 평균, (2) **혼합비율 + 클래스별 확률 보정**을 **같은 외부 Fold**에서 비교합니다.

- 이 노트북은 **독립 실행형**입니다. `steel.ipynb`의 셀을 모두 실행할 필요가 없습니다.
- **`Faults.NNA`와 `Faults27x7_var`** 파일을 이 노트북과 같은 폴더에 넣고 실행하세요.
- 데이터 20%는 기존 실험과 같이 테스트용으로 **보관만 하고 사용하지 않습니다**. 이미 기존 노트북에서 테스트 점수를 살펴보았으므로, 그 결과를 토대로 튜닝하면 독립 테스트의 의미가 줄어듭니다.
- 중첩 CV를 통해 보정 배율은 **외부 검증 Fold를 사용하지 않고** 고릅니다. 단, ET/LGBM 기본 파라미터는 기존 `steel.ipynb`의 개발셋 튜닝 결과를 그대로 가져왔으므로, 완전히 독립적인 Nested CV 성능 추정은 아닙니다.
- 시간이 충분하면 `N_REPEATS=3`으로 바꿔 안정성을 다시 검증하세요.

In [1]:
from pathlib import Path
from itertools import product
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
N_REPEATS = 1   # 먼저 1회(외부 3-Fold) 실행하고, 필요하면 3으로 변경
N_SPLITS = 3

root = Path('.')
cols = [s.strip() for s in (root / 'Faults27x7_var').read_text(encoding='utf-8').splitlines() if s.strip()]
raw = pd.read_csv(root / 'Faults.NNA', sep=r'\s+', header=None, names=cols)
assert raw.shape == (1941, 34), f'예상하지 못한 데이터 크기: {raw.shape}'
labels = list(raw.columns[27:34])
y_full = LabelEncoder().fit_transform(raw[labels].idxmax(axis=1))
X_full = raw.iloc[:, :27].drop(columns='TypeOfSteel_A300')  # steel.ipynb와 동일한 26개 변수

idx_dev, idx_test = train_test_split(
    np.arange(len(y_full)), test_size=0.2, stratify=y_full, random_state=RANDOM_STATE
)
X_dev, y_dev = X_full.iloc[idx_dev].reset_index(drop=True), y_full[idx_dev]
X_holdout, y_holdout = X_full.iloc[idx_test], y_full[idx_test]  # 이번 실험에서 절대 사용하지 않음
class_names = LabelEncoder().fit(raw[labels].idxmax(axis=1)).classes_.tolist()
print('개발 데이터:', X_dev.shape, '| 미사용 테스트:', X_holdout.shape)
print('결함 유형:', class_names)

# steel.ipynb의 26번 셀에 저장된 최적 파라미터
et = ExtraTreesClassifier(
    random_state=42, n_jobs=1, n_estimators=500, min_samples_split=5,
    min_samples_leaf=1, max_features=0.8, max_depth=None,
    class_weight='balanced_subsample'
)
lgb = LGBMClassifier(
    objective='multiclass', random_state=42, n_jobs=1, verbose=-1,
    subsample_freq=1, subsample=0.8, reg_lambda=0,
    num_leaves=22, n_estimators=700, min_child_samples=5,
    max_depth=-1, learning_rate=0.05, colsample_bytree=0.6,
    class_weight=None
)

def make_stack():
    return StackingClassifier(
        estimators=[('et', clone(et)), ('lgb', clone(lgb))],
        final_estimator=LogisticRegression(max_iter=2000),
        stack_method='predict_proba',
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=1),
        n_jobs=1
    )

def macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)


개발 데이터: (1552, 26) | 미사용 테스트: (389, 26)
결함 유형: ['Bumps', 'Dirtiness', 'K_Scatch', 'Other_Faults', 'Pastry', 'Stains', 'Z_Scratch']


In [2]:
# 클래스별 보정 후보는 이전 오분류 분석에서 중요한 3개 결함만 작게 조정합니다.
# 전체 7개 클래스를 큰 범위로 탐색하면 데이터에 과적합할 가능성이 높습니다.
BUMP = class_names.index('Bumps')
OTHER = class_names.index('Other_Faults')
PASTRY = class_names.index('Pastry')

ALPHAS = [0.25, 0.50, 0.75]  # alpha*ET + (1-alpha)*LightGBM
SCALE_GRID = list(product(
    [0.85, 0.95, 1.0],   # Bumps
    [0.9, 1.0, 1.1],     # Other_Faults
    [1.0, 1.15, 1.3],    # Pastry
))

def weights_for(combo, n_classes):
    weights = np.ones(n_classes)
    weights[[BUMP, OTHER, PASTRY]] = combo
    return weights

def blend(pe, pl, alpha, weights=None):
    P = alpha * pe + (1.0 - alpha) * pl
    if weights is not None:
        P = P * weights
    return P.argmax(axis=1)

def choose_calibration(Xa, ya, seed):
    # 학습 데이터 내부에서만 OOF 확률 생성. 외부 Fold의 정답/예측값은 사용하지 않음.
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    e_oof = cross_val_predict(clone(et), Xa, ya, cv=inner, method='predict_proba', n_jobs=1)
    l_oof = cross_val_predict(clone(lgb), Xa, ya, cv=inner, method='predict_proba', n_jobs=1)

    default_score = macro(ya, blend(e_oof, l_oof, 0.5))
    best = {'objective': default_score, 'alpha': 0.5, 'scale': np.ones(len(class_names)),
            'inner_f1': default_score}
    for a in ALPHAS:
        for combo in SCALE_GRID:
            weights = weights_for(combo, len(class_names))
            score = macro(ya, blend(e_oof, l_oof, a, weights))
            # 복잡한 보정 대신 간단한 설정을 선호하도록 약한 벌점 추가
            penalty = 0.002 * (abs(a - 0.5) + np.sum(np.abs(weights - 1.0)))
            if score - penalty > best['objective']:
                best = {'objective': score - penalty, 'alpha': a,
                        'scale': weights, 'inner_f1': score}

    # 교차검증 내부에서 개선이 충분하지 않으면 보정 없이 진행
    if best['inner_f1'] - default_score < 0.005:
        best = {'objective': default_score, 'alpha': 0.5,
                'scale': np.ones(len(class_names)), 'inner_f1': default_score}
    return best


In [3]:
outer = RepeatedStratifiedKFold(
    n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=2027
)
rows = []

for fold, (train_idx, val_idx) in enumerate(outer.split(X_dev, y_dev), 1):
    Xa, Xv = X_dev.iloc[train_idx], X_dev.iloc[val_idx]
    ya, yv = y_dev[train_idx], y_dev[val_idx]

    choice = choose_calibration(Xa, ya, seed=300 + fold)

    # 검증 데이터에 대한 예측은 외부 Fold 학습 데이터만으로 만든 모델을 사용
    pe = clone(et).fit(Xa, ya).predict_proba(Xv)
    pl = clone(lgb).fit(Xa, ya).predict_proba(Xv)
    stack = make_stack().fit(Xa, ya)

    fold_result = {
        'Fold': fold,
        'ET': macro(yv, pe.argmax(axis=1)),
        'LightGBM': macro(yv, pl.argmax(axis=1)),
        '50:50 소프트보팅': macro(yv, blend(pe, pl, 0.5)),
        '스태킹': macro(yv, stack.predict(Xv)),
        '혼합비율+확률보정': macro(yv, blend(pe, pl, choice['alpha'], choice['scale'])),
        '선택 ET 비중': choice['alpha'],
        '선택 배율(Bumps,Other,Pastry)': tuple(np.round(choice['scale'][[BUMP, OTHER, PASTRY]], 2)),
    }
    rows.append(fold_result)
    print(f'외부 Fold {fold} 완료 | 스태킹={fold_result["스태킹"]:.4f} | '
          f'보정={fold_result["혼합비율+확률보정"]:.4f}')

results = pd.DataFrame(rows)
score_cols = ['ET','LightGBM','50:50 소프트보팅','스태킹','혼합비율+확률보정']
print('\n[동일한 외부 Fold 기준 성능 비교]')
display(results[['Fold'] + score_cols].round(4))
summary = pd.DataFrame({'평균 Macro F1': results[score_cols].mean(),
                        '표준편차': results[score_cols].std(ddof=0)})
display(summary.sort_values('평균 Macro F1', ascending=False).round(4))
print('\n[보정값 선택 내역]')
display(results[['Fold','선택 ET 비중','선택 배율(Bumps,Other,Pastry)']])

winner = summary['평균 Macro F1'].idxmax()
print(f'\n현재 실험의 외부 검증 평균 1위: {winner}')
print('주의: 이 실험에서 성능이 가장 높게 나온 방식을 최종 채택할 때는 추가 독립 검증이 권장됩니다.')


외부 Fold 1 완료 | 스태킹=0.8238 | 보정=0.8026
외부 Fold 2 완료 | 스태킹=0.8193 | 보정=0.8082
외부 Fold 3 완료 | 스태킹=0.8401 | 보정=0.8202

[동일한 외부 Fold 기준 성능 비교]


,Fold,ET,LightGBM,50:50 소프트보팅,스태킹,혼합비율+확률보정
0,1,0.8308,0.8010,0.8074,0.8238,0.8026
1,2,0.8133,0.8121,0.8084,0.8193,0.8082
2,3,0.7986,0.8349,0.8419,0.8401,0.8202


,평균 Macro F1,표준편차
스태킹,0.8277,0.0090
50:50 소프트보팅,0.8193,0.0160
LightGBM,0.8160,0.0141
ET,0.8142,0.0131
혼합비율+확률보정,0.8103,0.0073



[보정값 선택 내역]


,Fold,선택 ET 비중,"선택 배율(Bumps,Other,Pastry)"
0,1,0.50,"(1.0, 1.1, 1.3)"
1,2,0.50,"(1.0, 1.1, 1.15)"
2,3,0.75,"(0.95, 1.0, 1.3)"



현재 실험의 외부 검증 평균 1위: 스태킹
주의: 이 실험에서 성능이 가장 높게 나온 방식을 최종 채택할 때는 추가 독립 검증이 권장됩니다.


## 해석 가이드

- **0.835 이상**: 외부 Fold 평균 Macro F1이 0.835를 넘는지 먼저 확인하세요. 한 Fold만 넘는 것은 충분하지 않습니다.
- **확률 보정 후 감소**: 보정 기법을 최종 모델에서 빼는 것이 타당합니다.
- **차이가 작은 경우**: 표준편차와 Fold별 승리 횟수를 함께 보고 간단한 모델을 우선하세요.
- **중요:** 이 노트북은 기존 개발 데이터로 선택한 ET/LGBM 파라미터를 사용하므로, 완전한 선택 편향 제거를 원한다면 **모델 하이퍼파라미터 탐색부터 외부 Fold 안에 넣은 Nested CV**를 별도로 해야 합니다. 기존에 이미 성능을 보고 사용한 20% 테스트셋은 추가 튜닝을 위한 검증 데이터로 쓰지 않는 것이 좋습니다.


## 2단계. CatBoost를 추가한 3모델 스태킹 실험

**이전 외부 3-Fold와 동일하게 비교합니다.** 기존 `results`의 스태킹 점수를 재사용하므로 반드시 이전 셀을 먼저 실행하세요. `X_holdout`, `y_holdout`은 사용하지 않습니다.

- 기존 스태킹: ET + LightGBM → LogisticRegression
- 신규 스태킹: ET + LightGBM + CatBoost → LogisticRegression
- 신규 스태킹 안의 3-Fold는 **외부 학습 데이터 내부에서만** 수행됩니다.
- 속도를 위해 CatBoost는 350회 반복, 내부 CV는 3-Fold로 시작합니다. 이것은 기존 최종 스태킹 내부 5-Fold와 구조가 조금 다르므로 **정확한 개선 원인을 더 확인하려면 신규 스태킹만 내부 5-Fold로 재검증**해야 합니다.
- 이 결과는 탐색적 개발 데이터 검증입니다. 여기서 가장 높은 후보를 골랐다면 새로운 분할로 재평가해야 합니다.

In [4]:
# 필요한 경우 한 번만 설치하세요.
# %pip install catboost

from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.base import clone
from sklearn.metrics import f1_score
from IPython.display import display
import numpy as np
import pandas as pd

cat = CatBoostClassifier(
    iterations=350,
    depth=5,
    learning_rate=0.035,
    l2_leaf_reg=5,
    loss_function='MultiClass',
    random_seed=42,
    verbose=False,
    thread_count=1,
    allow_writing_files=False,
)

def make_stack_3():
    return StackingClassifier(
        estimators=[
            ('et', clone(et)),
            ('lgb', clone(lgb)),
            ('cat', clone(cat)),
        ],
        final_estimator=LogisticRegression(max_iter=2000, C=1.0),
        stack_method='predict_proba',
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=1),
        n_jobs=1,
    )

# 기존 실험의 바깥 검증과 동일한 폴드를 생성
outer2 = RepeatedStratifiedKFold(n_splits=3, n_repeats=1, random_state=2027)
new_rows = []

for fold, (tr, va) in enumerate(outer2.split(X_dev, y_dev), 1):
    Xa, Xv = X_dev.iloc[tr], X_dev.iloc[va]
    ya, yv = y_dev[tr], y_dev[va]

    # 아래 fit은 외부 학습 Fold에서만 실시합니다.
    m3 = make_stack_3()
    m3.fit(Xa, ya)
    p3 = m3.predict(Xv)
    f3 = f1_score(yv, p3, average='macro', zero_division=0)

    # 이전 결과는 동일한 외부 Fold이므로 그대로 비교할 수 있습니다.
    f2 = float(results.iloc[fold - 1]['스태킹'])
    new_rows.append({
        'Fold': fold,
        '기존 2모델 스태킹': f2,
        '새로운 3모델 스태킹': f3,
        '개선량': f3 - f2,
    })
    print(f'Fold {fold} | 기존={f2:.4f} | 3모델={f3:.4f} | 차이={f3-f2:+.4f}', flush=True)

compare3 = pd.DataFrame(new_rows)
print('\n[동일 Fold 성능 비교]')
display(compare3.round(4))

cols3 = ['기존 2모델 스태킹', '새로운 3모델 스태킹']
summary3 = pd.DataFrame({
    '평균 Macro F1': compare3[cols3].mean(),
    '표준편차': compare3[cols3].std(ddof=0),
})
print('\n[평균과 표준편차]')
display(summary3.round(4))

avg2 = float(summary3.loc['기존 2모델 스태킹', '평균 Macro F1'])
avg3 = float(summary3.loc['새로운 3모델 스태킹', '평균 Macro F1'])
print(f'\n기존 평균: {avg2:.4f} → 새 평균: {avg3:.4f} ({avg3-avg2:+.4f})')
print('목표 0.835 달성 여부:', '달성 후보' if avg3 >= 0.835 else '미달성')
print('3모델 우세 Fold:', int((compare3['개선량'] > 0).sum()), '/ 3')
print('주의: 아직 최종 테스트 성능이 아니며, 개선이 확인되면 별도의 새로운 CV 분할에서 다시 평가하세요.')

Fold 1 | 기존=0.8238 | 3모델=0.8223 | 차이=-0.0014
Fold 2 | 기존=0.8193 | 3모델=0.8117 | 차이=-0.0076
Fold 3 | 기존=0.8401 | 3모델=0.8398 | 차이=-0.0003

[동일 Fold 성능 비교]


,Fold,기존 2모델 스태킹,새로운 3모델 스태킹,개선량
0,1,0.8238,0.8223,-0.0014
1,2,0.8193,0.8117,-0.0076
2,3,0.8401,0.8398,-0.0003



[평균과 표준편차]


,평균 Macro F1,표준편차
기존 2모델 스태킹,0.8277,0.0090
새로운 3모델 스태킹,0.8246,0.0116



기존 평균: 0.8277 → 새 평균: 0.8246 (-0.0031)
목표 0.835 달성 여부: 미달성
3모델 우세 Fold: 0 / 3
주의: 아직 최종 테스트 성능이 아니며, 개선이 확인되면 별도의 새로운 CV 분할에서 다시 평가하세요.
